In [69]:
import itertools
import nltk
from nltk.corpus import wordnet
import random
from tqdm import tqdm

In [75]:
def generate_sentence_variants(structure, n1, n2, v1, adj1, adj2, adv1):
    """
    Generates all valid combinations of modifiers for:
    1. The [adj1] N1 [rel] the [adj2] N2 [adv1] V1
    2. The [adj2] N2 [adv1] V1 the [adj1] N1
    """
    
    # Define the possible states for each modifier (Either the word or an empty string)
    modifiers = {
        'adj1': [adj1, ""],
        'adj2': [adj2, ""],
        'adv1': [adv1, ""],
        'rel': ["that", "who", ""]
    }
    
    # Create the Cartesian product of all variations
    keys = modifiers.keys()
    combinations = list(itertools.product(*modifiers.values()))
    
    all_sentences = set()

    for combo in combinations:
        # Map combo back to keys for readability
        d = dict(zip(keys, combo))
        if structure == 1:
            # Structure 1: The [adj1] N1 [rel] the [adj2] N2 [adv1] V1
            s = f"The {d['adj1']} {n1} {d['rel']} the {d['adj2']} {n2} {d['adv1']} {v1}"
        elif structure == 2:
        # Structure 2: The [adj2] N2 [adv1] V1 the [adj1] N1
            s = f"The {d['adj2']} {n2} {d['adv1']} {v1} the {d['adj1']} {n1}"
        
        # Clean up double spaces and strip
        clean_s = " ".join(s.split()).strip()
        all_sentences.add(clean_s.capitalize() + ".")

    return all_sentences

# --- Example Usage ---
# n1, n2, v1 = "cake", "chef", "baked"
# adj1, adj2, adv1 = "sweet", "famous", "quickly"

# results = generate_sentence_variants(n1, n2, v1, adj1, adj2, adv1)
# for s in sorted(results):
#     print(s)

In [76]:
n1, n2, v1 = "boy", "girl", "likes"
adj1, adj2, adv1 = "big", "beautiful", "really"

results = generate_sentence_variants(1,n1, n2, v1, adj1, adj2, adv1)
gen = list(sorted(results))
for s in gen:
    print(s)

The big boy that the beautiful girl likes.
The big boy that the beautiful girl really likes.
The big boy that the girl likes.
The big boy that the girl really likes.
The big boy the beautiful girl likes.
The big boy the beautiful girl really likes.
The big boy the girl likes.
The big boy the girl really likes.
The big boy who the beautiful girl likes.
The big boy who the beautiful girl really likes.
The big boy who the girl likes.
The big boy who the girl really likes.
The boy that the beautiful girl likes.
The boy that the beautiful girl really likes.
The boy that the girl likes.
The boy that the girl really likes.
The boy the beautiful girl likes.
The boy the beautiful girl really likes.
The boy the girl likes.
The boy the girl really likes.
The boy who the beautiful girl likes.
The boy who the beautiful girl really likes.
The boy who the girl likes.
The boy who the girl really likes.


for each combination : generates 24 ORC, and 8 right branching 

In [74]:
len(gen)

32

In [86]:
from lemminflect import getInflection
import itertools

def conjugate_verb(lemma, is_plural):
    """Conjugates to Present Tense: 'likes' (singular) vs 'like' (plural)."""
    # VBP is non-3rd person present (plural/I/you), VBZ is 3rd person singular
    tag = 'VBP' if is_plural else 'VBZ'
    inflection = getInflection(lemma, tag=tag)
    return inflection[0] if inflection else lemma

def pluralize_noun(lemma):
    """Conjugates noun to plural (NNS)."""
    inflection = getInflection(lemma, tag='NNS')
    return inflection[0] if inflection else lemma + 's'

import itertools
from lemminflect import getInflection

import itertools

def generate_sentence_variants(structure, n1, n2, v1_sing, v1_plur, adj1, adj2, adv1, n1_is_plural=False, n2_is_plural=False):
    # 1. Handle Noun Number (using simple 's' or lemminflect if you prefer)
    # If you have specific plural nouns (like 'children'), pass them in or use pluralize_noun
    current_n1 = n1 + "s" if n1_is_plural else n1 
    current_n2 = n2 + "s" if n2_is_plural else n2
    
    # 2. STRICT VERB AGREEMENT: Verb always follows N2
    current_v1 = v1_plur if n2_is_plural else v1_sing
    
    modifiers = {
        'adj1': [adj1, ""],
        'adj2': [adj2, ""],
        'adv1': [adv1, ""],
        'rel': ["that", "who", ""]
    }
    
    keys = modifiers.keys()
    combinations = list(itertools.product(*modifiers.values()))
    all_sentences = set()

    for combo in combinations:
        d = dict(zip(keys, combo))
        if structure == 1:
            # The [adj1] N1 [rel] the [adj2] N2 [adv1] V1
            s = f"The {d['adj1']} {current_n1} {d['rel']} the {d['adj2']} {current_n2} {d['adv1']} {current_v1}"
        elif structure == 2:
            # The [adj2] N2 [adv1] V1 the [adj1] N1
            s = f"The {d['adj2']} {current_n2} {d['adv1']} {current_v1} the {d['adj1']} {current_n1}"
        
        all_sentences.add(" ".join(s.split()).strip().capitalize() + ".")
        
    return all_sentences

In [136]:
def generate_sentence_variants(structure, n1, n2, v1_sing, v1_plur, adj1, adj2, adv1, n1_is_plural=False, n2_is_plural=False):
    current_n1 = pluralize_noun(n1) if n1_is_plural else n1 
    current_n2 = pluralize_noun(n2) if n2_is_plural else n2
    current_v1 = v1_plur if n2_is_plural else v1_sing
    
    modifiers = {
        'adj1': [adj1, ""],
        'adj2': [adj2, ""],
        'adv1': [adv1, ""],
        'rel': ["that", "who", ""]
    }
    
    keys = modifiers.keys()
    combinations = list(itertools.product(*modifiers.values()))
    
    # We return a list of tuples: (core_string, n1_is_plural)
    cores = []

    for combo in combinations:
        d = dict(zip(keys, combo))
        if structure == 1:
            s_core = f"The {d['adj1']} {current_n1} {d['rel']} the {d['adj2']} {current_n2} {d['adv1']} {current_v1}"
        else:
            s_core = f"The {d['adj2']} {current_n2} {d['adv1']} {current_v1} the {d['adj1']} {current_n1}"
        
        clean_core = " ".join(s_core.split()).strip()
        cores.append((clean_core, n1_is_plural, structure))
        
    return cores

In [137]:
def generate_all_scenarios(structure, n1, n2, v1_sing, v1_plur, adj1, adj2, adv1):
    scenarios = [
        (False, False), # Sing N1, Sing N2 -> v1_sing
        (True, True),   # Plur N1, Plur N2 -> v1_plur
        (False, True),  # Sing N1, Plur N2 -> v1_plur (Incongruent)
        (True, False)   # Plur N1, Sing N2 -> v1_sing (Incongruent)
    ]
    
    master_set = set()
    for n1_p, n2_p in scenarios:
        variants = generate_sentence_variants(
            structure, n1, n2, v1_sing, v1_plur, adj1, adj2, adv1, n1_p, n2_p
        )
        master_set.update(variants)
    return master_set

In [140]:
def generate_full_corpus(n1_opts, n2_opts, v_opts, adj1_opts, adj2_opts, adv1_opts):
    # This set will store (core_string, n1_is_plural, structure)
    # Because it's a set of tuples, it will catch duplicates in the core!
    unique_cores = set()
    
    word_combinations = itertools.product(n1_opts, n2_opts, v_opts, adj1_opts, adj2_opts, adv1_opts)
    
    for n1, n2, v_pair, adj1, adj2, adv1 in word_combinations:
        v_sing, v_plur = v_pair
        
        # We only care about Structure 1 (ORC) based on your loop
        for struct in [1]:
            for n1_p, n2_p in itertools.product([False, True], [False, True]):
                scenarios = generate_sentence_variants(
                    struct, n1, n2, v_sing, v_plur, adj1, adj2, adv1, n1_p, n2_p
                )
                unique_cores.update(scenarios)

    # NOW we add the random continuations to the unique cores
    final_sentences = []
    
    cont_sing = ["is eating an apple", "is watching a movie", "is reading a book"]
    cont_plur = ["are eating an apple", "are watching a movie", "are reading a book"]

    for core, is_plural, struct in unique_cores:
        if struct == 1:
            tail = random.choice(cont_plur) if is_plural else random.choice(cont_sing)
            full_sent = f"{core} {tail}."
        else:
            full_sent = f"{core}."
            
        final_sentences.append(full_sent.capitalize())

    return final_sentences

In [142]:
# 1. Define the 3-word vocabulary
n1_opts = ["boy", "student", "doctor", "artist", "athlete"]
n2_opts = ["girl", "child", "pilot", "scientist", "engineer"] # 'child' requires an irregular plural check
v_opts = [("visits", "visit"), ("helps", "help"), ("avoids", "avoid"), ("follows", "follow"), ("greets", "greet")]
adj1_opts = ["big", "tall", "young", "strong", "kind"]
adj2_opts = ["beautiful", "smart", "brave", "famous", "honest"]
adv1_opts = ["really", "secretly", "always", "often", "rarely"]

# 2. Update Noun Pluralization for irregulars
def pluralize_noun(noun):
    irregulars = {"child": "children", "man": "men", "woman": "women"}
    if noun in irregulars:
        return irregulars[noun]
    return noun + "s"

# 3. Generate the massive corpus
# Note: This might take a few seconds to run due to the 93k+ iterations
final_corpus = generate_full_corpus(
    n1_opts, n2_opts, v_opts, 
    adj1_opts, adj2_opts, adv1_opts
)

print(f"Total Sentences in Corpus: {len(final_corpus)}")

Total Sentences in Corpus: 324000


In [143]:
final_corpus

['The doctors that the brave children always visit are reading a book.',
 'The strong students the smart girls really follow are watching a movie.',
 'The strong artists the pilots rarely greet are reading a book.',
 'The young athlete that the pilot often avoids is eating an apple.',
 'The artists the honest girls secretly help are eating an apple.',
 'The kind boys the beautiful scientists rarely greet are reading a book.',
 'The strong athlete that the brave pilots always visit is reading a book.',
 'The kind students who the pilot greets are watching a movie.',
 'The big student that the beautiful children avoid is eating an apple.',
 'The tall boys who the famous girls rarely greet are eating an apple.',
 'The big boys who the famous engineers always help are eating an apple.',
 'The tall doctor who the smart scientist often greets is watching a movie.',
 'The strong boy who the famous pilot secretly helps is eating an apple.',
 'The doctors that the brave children secretly follow

In [144]:
import itertools
import random

def generate_wh_variants(n1, n2, v_pair, adj1, adj2, adv1, n1_plur=False, n2_plur=False):
    v_base, v_ing = v_pair
    
    # 1. Prepare Nouns
    curr_n1 = pluralize_noun(n1) if n1_plur else n1
    curr_n2 = pluralize_noun(n2) if n2_plur else n2
    
    # 2. Tense/Auxiliary Mapping based on N1
    # Format: (Aux, Verb_to_use)
    tenses = [
        ("did", v_base),                                # Past
        ("does" if not n1_plur else "do", v_base),      # Present
        ("is" if not n1_plur else "are", v_ing)         # Progressive
    ]
    
    # 3. Modifier Toggles (On/Off)
    modifiers = {
        'a1': [adj1, ""],
        'a2': [adj2, ""],
        'ad1': [adv1, ""]
    }
    
    combos = list(itertools.product(*modifiers.values()))
    questions = []

    for aux, v_final in tenses:
        for a1, a2, ad1 in combos:
            # Template 1: What
            # What [aux] the [adj1] [N1] [adv1] [v]?
            q1 = f"What {aux} the {a1} {curr_n1} {ad1} {v_final}?"
            questions.append(" ".join(q1.split()).capitalize())
            
            # Template 2: Which
            # Which [adj2] [N2] [aux] the [adj1] [N1] [adv1] [v]?
            q2 = f"Which {a2} {curr_n2} {aux} the {a1} {curr_n1} {ad1} {v_final}?"
            questions.append(" ".join(q2.split()).capitalize())
            
    return questions

In [149]:
import itertools

def generate_wh_variants(n1, n2, v_pair, adj1, adj2, adv1, n1_plur=False, n2_plur=False):
    v_base, v_ing = v_pair
    
    # Prepare Nouns
    curr_n1 = pluralize_noun(n1) if n1_plur else n1
    curr_n2 = pluralize_noun(n2) if n2_plur else n2
    
    # Modifier options
    mods = {'a1': [adj1, ""], 'a2': [adj2, ""], 'ad1': [adv1, ""]}
    combos = list(itertools.product(*mods.values()))
    
    questions = []

    # Iterate through both nouns acting as the Subject
    for subject_info in [("N1", curr_n1, n1_plur), ("N2", curr_n2, n2_plur)]:
        sub_label, sub_noun, sub_is_plur = subject_info
        
        # Determine auxiliaries based on the CURRENT subject
        tenses = [
            ("did", v_base), 
            ("does" if not sub_is_plur else "do", v_base),
            ("is" if not sub_is_plur else "are", v_ing)
        ]

        for aux, v_final in tenses:
            for a1, a2, ad1 in combos:
                # 1. Template: What (Subject is N1 or N2)
                # We use the adjective corresponding to the current subject
                current_adj = a1 if sub_label == "N1" else a2
                q_what = f"What {aux} the {current_adj} {sub_noun} {ad1} {v_final}?"
                questions.append(" ".join(q_what.split()).capitalize())

                # 2. Template: Which (N1 is Subject, N2 is the Wh-Object)
                # Only run this when N1 is subject to avoid redundant 'Which' questions
                if sub_label == "N1":
                    q_which = f"Which {a2} {curr_n2} {aux} the {a1} {sub_noun} {ad1} {v_final}?"
                    questions.append(" ".join(q_which.split()).capitalize())
            
    return questions

In [150]:
n1 = "boy"
n2 = "girl"
v_pair = ("like", "liking")
adj1 = "big"
adj2 = "beautiful"

In [151]:
q = generate_wh_variants(n1, n2, v_pair, adj1, adj2, adv1, n1_plur=False, n2_plur=False)
for question in q:
    print(question)

What did the big boy really like?
Which beautiful girl did the big boy really like?
What did the big boy like?
Which beautiful girl did the big boy like?
What did the big boy really like?
Which girl did the big boy really like?
What did the big boy like?
Which girl did the big boy like?
What did the boy really like?
Which beautiful girl did the boy really like?
What did the boy like?
Which beautiful girl did the boy like?
What did the boy really like?
Which girl did the boy really like?
What did the boy like?
Which girl did the boy like?
What does the big boy really like?
Which beautiful girl does the big boy really like?
What does the big boy like?
Which beautiful girl does the big boy like?
What does the big boy really like?
Which girl does the big boy really like?
What does the big boy like?
Which girl does the big boy like?
What does the boy really like?
Which beautiful girl does the boy really like?
What does the boy like?
Which beautiful girl does the boy like?
What does the boy 

In [152]:
len(q)

72

In [153]:

import itertools

def generate_all_scenarios(n1, n2, v_pair, adj1, adj2, adv1):
    # The 4 Number Scenarios: (n1_is_plural, n2_is_plural)
    scenarios = [
        (False, False), # SS
        (True, True),   # PP
        (False, True),  # SP
        (True, False)   # PS
    ]
    
    all_variants = []
    
    for n1_p, n2_p in scenarios:
        # Generate variants for this specific number combination
        variants = generate_wh_variants(
            n1, n2, v_pair, adj1, adj2, adv1, 
            n1_plur=n1_p, n2_plur=n2_p
        )
        all_variants.extend(variants)
        
    return all_variants

def generate_wh_variants(n1, n2, v_pair, adj1, adj2, adv1, n1_plur, n2_plur):
    v_base, v_ing = v_pair
    curr_n1 = pluralize_noun(n1) if n1_plur else n1
    curr_n2 = pluralize_noun(n2) if n2_plur else n2
    
    # Modifiers
    mods = {'a1': [adj1, ""], 'a2': [adj2, ""], 'ad1': [adv1, ""]}
    combos = list(itertools.product(*mods.values()))
    
    results = []

    # Case A: N1 is the Subject (What/Which)
    # Aux agrees with N1
    aux_list_n1 = [
        ("did", v_base), 
        ("does" if not n1_plur else "do", v_base),
        ("is" if not n1_plur else "are", v_ing)
    ]

    for aux, v_fin in aux_list_n1:
        for a1, a2, ad1 in combos:
            # Template 1: What (N1 is Subject)
            q_what_n1 = f"What {aux} the {a1} {curr_n1} {ad1} {v_fin}?"
            results.append(" ".join(q_what_n1.split()).capitalize())
            
            # Template 2: Which (N1 is Subject, N2 is Object)
            q_which = f"Which {a2} {curr_n2} {aux} the {a1} {curr_n1} {ad1} {v_fin}?"
            results.append(" ".join(q_which.split()).capitalize())

    # Case B: N2 is the Subject (What only)
    # Aux agrees with N2
    aux_list_n2 = [
        ("did", v_base), 
        ("does" if not n2_plur else "do", v_base),
        ("is" if not n2_plur else "are", v_ing)
    ]

    for aux, v_fin in aux_list_n2:
        for a1, a2, ad1 in combos:
            # Template 1: What (N2 is Subject)
            q_what_n2 = f"What {aux} the {a2} {curr_n2} {ad1} {v_fin}?"
            results.append(" ".join(q_what_n2.split()).capitalize())
            
    return results

In [154]:
# Nouns (Human/Social Roles)
n1_opts = ["student", "doctor", "pilot", "officer", "athlete", 
           "artist", "scientist", "teacher", "lawyer", "judge"]

n2_opts = ["child", "girl", "boy", "patient", "client", 
           "tourist", "driver", "neighbor", "guest", "worker"]

# Verbs (Action/Social - Transitive)
# Format: (Base Form, Progressive -ing Form)
v_opts = [
    ("visit", "visiting"),
    ("help", "helping"),
    ("greet", "greeting"),
    ("follow", "following"),
    ("avoid", "avoiding"),
    ("call", "calling"),
    ("observe", "observing"),
    ("assist", "assisting"),
    ("meet", "meeting"),
    ("see", "seeing")
]

# Adjectives (Personality/Status)
adj1_opts = ["young", "tall", "smart", "brave", "kind", 
             "famous", "honest", "strong", "busy", "calm"]

adj2_opts = ["creative", "serious", "friendly", "quiet", "active", 
             "careful", "proud", "gentle", "fair", "eager"]

# Adverbs (Frequency - Placement is before the verb)
adv1_opts = ["often", "always", "rarely", "secretly", "usually", 
             "quickly", "quietly", "clearly", "daily", "never"]

In [155]:
import itertools

# Use the pluralize_noun function we defined earlier
def pluralize_noun(noun):
    irregulars = {"child": "children", "man": "men", "woman": "women"}
    return irregulars.get(noun, noun + "s")

def generate_final_corpus_to_file(filename):
    word_products = itertools.product(n1_opts, n2_opts, v_opts, adj1_opts, adj2_opts, adv1_opts)
    
    with open(filename, "w") as f:
        for n1, n2, v_pair, adj1, adj2, adv1 in word_products:
            # Generate all 4 number scenarios
            for n1_p, n2_p in itertools.product([False, True], [False, True]):
                
                # Fetch all grammatical variants (What/Which/Tenses/Modifiers)
                batch = generate_wh_variants(n1, n2, v_pair, adj1, adj2, adv1, n1_p, n2_p)
                
                # Write to file immediately to save RAM
                for sentence in batch:
                    f.write(sentence + "\n")

# Run the generation
# generate_final_corpus_to_file("wh_questions_corpus.txt")

In [156]:
generate_final_corpus_to_file("testwhquestions.txt")

In [157]:
def generate_declarative_variants(n1, n2, v_tuple, adj1, adj2, adv1, n1_plur, n2_plur):
    # v_tuple example: ("visit", "visited", "visiting")
    v_base, v_past, v_ing = v_tuple
    
    curr_n1 = pluralize_noun(n1) if n1_plur else n1
    curr_n2 = pluralize_noun(n2) if n2_plur else n2
    
    # 1. Handle Verb Agreement for the 3 Tenses
    # Present Tense Agreement
    v_pres = v_base if n1_plur else (v_base + "s" if not v_base.endswith('s') else v_base + "es")
    # Progressive Agreement
    aux_prog = "are" if n1_plur else "is"
    
    tenses = [
        v_past,                 # Past: "The doctor visited..."
        v_pres,                 # Present: "The doctor visits..."
        f"{aux_prog} {v_ing}"   # Progressive: "The doctor is visiting..."
    ]
    
    # 2. Modifier Combinations (2^3 = 8)
    mods = {'a1': [adj1, ""], 'a2': [adj2, ""], 'ad1': [adv1, ""]}
    combos = list(itertools.product(*mods.values()))
    
    declaratives = []
    for verb_phrase in tenses:
        for a1, a2, ad1 in combos:
            # Structure: The [adj1] N1 [adv1] [Verb] the [adj2] N2.
            s = f"The {a1} {curr_n1} {ad1} {verb_phrase} the {a2} {curr_n2}."
            declaratives.append(" ".join(s.split()).capitalize())
            
    return declaratives

In [158]:
def pluralize_noun(noun):
    irregulars = {"child": "children", "man": "men", "woman": "women"}
    return irregulars.get(noun, noun + "s")

def generate_final_corpus_to_file(filename):
    word_products = itertools.product(n1_opts, n2_opts, v_opts, adj1_opts, adj2_opts, adv1_opts)
    
    with open(filename, "w") as f:
        for n1, n2, v_pair, adj1, adj2, adv1 in word_products:
            # Generate all 4 number scenarios
            for n1_p, n2_p in itertools.product([False, True], [False, True]):
                
                # Fetch all grammatical variants (What/Which/Tenses/Modifiers)
                batch = generate_declarative_variants(n1, n2, v_pair, adj1, adj2, adv1, n1_p, n2_p)
                
                # Write to file immediately to save RAM
                for sentence in batch:
                    f.write(sentence + "\n")

In [160]:
# Format: (Base, Past, Progressive)
v_opts = [
    ("visit", "visited", "visiting"),
    ("help", "helped", "helping"),
    ("greet", "greeted", "greeting"),
    ("follow", "followed", "following"),
    ("avoid", "avoided", "avoiding"),
    ("call", "called", "calling"),
    ("observe", "observed", "observing"),
    ("assist", "assisted", "assisting"),
    ("meet", "met", "meeting"),
    ("see", "saw", "seeing")
]

In [161]:
generate_final_corpus_to_file("testdeclaratives.txt")

In [164]:
def check_vocab_coverage(file_path, options_lists):
    # Load your vocab into a set for O(1) lookup speed
    with open(file_path, 'r', encoding='utf-8') as f:
        vocab = {line.strip().lower() for line in f}

    missing = []
    
    # Flatten all your lists into one big check-list
    for category_name, words in options_lists.items():
        for word in words:
            # Handle tuples (like verb pairs)
            if isinstance(word, tuple):
                for sub_word in word:
                    if sub_word.lower() not in vocab:
                        missing.append(f"{category_name}: {sub_word}")
            else:
                if word.lower() not in vocab:
                    missing.append(f"{category_name}: {word}")

    if not missing:
        print("✅ All words exist in vocab.txt!")
    else:
        print("❌ Missing words found:")
        for item in missing:
            print(f"  - {item}")

# --- SETUP ---
my_options = {
    # Nouns (Human/Social Roles)
"n1_opts" = ["student", "doctor", "pilot", "officer", "athlete", 
           "artist", "scientist", "teacher", "lawyer", "judge"]

"n2_opts" = ["child", "girl", "boy", "patient", "client", 
           "tourist", "driver", "neighbor", "guest", "worker"]

# Verbs (Action/Social - Transitive)
# Format: (Base Form, Progressive -ing Form)
"v_opts" = [
    ("visit", "visiting"),
    ("help", "helping"),
    ("greet", "greeting"),
    ("follow", "following"),
    ("avoid", "avoiding"),
    ("call", "calling"),
    ("observe", "observing"),
    ("assist", "assisting"),
    ("meet", "meeting"),
    ("see", "seeing")
]

# Adjectives (Personality/Status)
"adj1_opts" = ["young", "tall", "smart", "brave", "kind", 
             "famous", "honest", "strong", "busy", "calm"]

"adj2_opts" = ["creative", "serious", "friendly", "quiet", "active", 
             "careful", "proud", "gentle", "fair", "eager"]

# Adverbs (Frequency - Placement is before the verb)
"adv1_opts" = ["often", "always", "rarely", "secretly", "usually", 
             "quickly", "quietly", "clearly", "daily", "never"]
}

# check_vocab_coverage("vocab.txt", my_options)

SyntaxError: cannot assign to literal here. Maybe you meant '==' instead of '='? (3723244061.py, line 30)

In [165]:
check_vocab_coverage("data/base_data/vocab.txt", my_options)

✅ All words exist in vocab.txt!


In [ ]:
§